<a href="https://colab.research.google.com/github/katjanieberle/sparks/blob/main/SPARKS_uplift.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Uplift Modeling & Causal Inference Project

## Welcome, Students!

As your lecturer and an experienced programmer, I've designed this project to deepen your understanding of Uplift Modeling and Causal Inference. This field is crucial for making data-driven decisions, especially in marketing, healthcare, and policy, by identifying customers or individuals who are most likely to respond positively to a specific treatment or intervention.

This project assumes you are familiar with Exploratory Data Analysis (EDA), supervised, and unsupervised learning models. You will work independently to implement and compare various uplift modeling techniques.

### Learning Objectives:
*   Understand the fundamental concepts of uplift modeling and its distinction from traditional prediction.
*   Implement and compare different uplift modeling approaches: S-Learner, T-Learner, X-Learner, and R-Learner.
*   Evaluate uplift models using appropriate metrics (e.g., Qini curve, AUUC).
*   Interpret model results and derive actionable business recommendations.
*   Develop robust and well-documented code.

### Project Structure:
This notebook is structured to guide you through the project. You will find descriptions, tasks, and code snippets. Some sections will require you to write your own code to complete the tasks.

## Setup and Dependencies

First, let's install and import the necessary libraries. We will primarily use `scikit-learn` for machine learning models, `pandas` and `numpy` for data manipulation, `matplotlib` and `seaborn` for visualization, and the `scikit-uplift` (`sklift`) library for uplift modeling.

In [10]:
# Install necessary libraries (if not already installed)
!pip install pandas numpy scikit-learn matplotlib seaborn scikit-uplift

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

from sklift.datasets import fetch_criteo
from sklift.models import SoloModel, TwoModels, ClassTransformation
from sklift.models.r_learner import RLearner # Corrected import for RLearner
from sklift.metrics import uplift_at_k, qini_curve, plot_qini_curve

print("Libraries imported successfully!")

ModuleNotFoundError: No module named 'sklift.models.r_learner'

## Dataset Description: Criteo Uplift Prediction Dataset

For this project, we will use the Criteo Uplift Prediction Dataset, which is readily available through `sklift`. This dataset is commonly used in uplift modeling research and provides a realistic scenario for understanding customer response to a marketing campaign.

Each row represents a customer, with features describing them, an indicator of whether they were in the treatment or control group, and a binary outcome indicating conversion.

**Dataset Components:**
*   `X`: DataFrame of customer features.
*   `y`: Series of binary conversion outcomes (0 or 1).
*   `t`: Series of binary treatment indicators (0 for control, 1 for treatment).

Let's load the dataset and examine its structure.

In [ ]:
# Load the Criteo Uplift Prediction dataset
criteo_data = fetch_criteo()

# Correctly access features, target, and treatment from the Bunch object
X = criteo_data.data
y = criteo_data.target
t = criteo_data.treatment

print("Features (X) head:")
display(X.head())

print("\nConversion (y) head:")
display(y.head())

print("\nTreatment (t) head:")
display(t.head())

print("\nDataset info for X:")
X.info()

print("\nTreatment and Conversion distribution:")
combined_df = pd.DataFrame({'treatment': t, 'conversion': y})
display(combined_df.groupby('treatment')['conversion'].value_counts(normalize=True))

## Task 1: Data Preprocessing and Exploratory Data Analysis (EDA)

Before building any models, it's crucial to understand your data. Perform thorough EDA to identify patterns, distributions, and potential issues. For uplift modeling, pay special attention to the distribution of treatment and control groups, and how features might influence conversion.

### Instructions:
1.  **Check for Missing Values:** Identify and handle any missing data in `X`, `y`, and `t`.
2.  **Descriptive Statistics:** Calculate and display descriptive statistics for `X`.
3.  **Visualize Distributions:**
    *   Plot histograms or kernel density estimates for numerical features in `X`.
    *   Visualize the distribution of `t` (treatment) and `y` (conversion).
4.  **Analyze Treatment and Control Groups:**
    *   Compare the conversion rates between the treatment and control groups directly.
    *   Examine if the distribution of features is similar between the treatment and control groups (covariate balance). This is important for causal inference assumptions.
5.  **Correlation Analysis:** Investigate correlations between features in `X` and the outcome `y`.

**Your Code Goes Here:**

In [3]:
# --- Your code for Task 1: Data Preprocessing and EDA ---

# Combine X, y, t into a single DataFrame for easier EDA
full_df_eda = pd.concat([X, y.rename('conversion'), t.rename('treatment')], axis=1)

# 1. Check for Missing Values
print("Missing values per column:")
display(full_df_eda.isnull().sum())

# Handle missing values (Example: fill with median for numerical features)
# Note: Criteo dataset is usually quite clean, but good to check.
for col in X.columns:
    if full_df_eda[col].dtype in ['float64', 'int64']:
        full_df_eda[col] = full_df_eda[col].fillna(full_df_eda[col].median())

print("\nMissing values after handling:")
display(full_df_eda.isnull().sum())

# 2. Descriptive Statistics
print("\nDescriptive statistics for features (X):")
display(X.describe())

# 3. Visualize Distributions
# Histograms for numerical features
for col in X.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(full_df_eda[col], kde=True)
    plt.title(f'Distribution of {col}')
    plt.show()

# Distribution of treatment and conversion
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.countplot(x='treatment', data=full_df_eda)
plt.title('Distribution of Treatment Group')
plt.subplot(1, 2, 2)
sns.countplot(x='conversion', data=full_df_eda)
plt.title('Distribution of Conversion Outcome')
plt.tight_layout()
plt.show()

# 4. Analyze Treatment and Control Groups
print("\nConversion rates by treatment group:")
display(full_df_eda.groupby('treatment')['conversion'].mean())

# Compare feature distributions between treatment and control (visual inspection)
for col in X.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(data=full_df_eda, x=col, hue='treatment', kde=True, common_norm=False)
    plt.title(f'Distribution of {col} by Treatment Group')
    plt.show()

# 5. Correlation Analysis
print("\nCorrelation matrix between features and conversion:")
display(full_df_eda[X.columns.tolist() + ['conversion']].corr(numeric_only=True).loc['conversion'][:-1])

NameError: name 'X' is not defined

## Task 2: Uplift Model Implementation (S-Learner, T-Learner, X-Learner, R-Learner)

Now, let's implement different meta-learners to estimate the Conditional Average Treatment Effect (CATE) or uplift. We will use `sklift`'s implementations, which provide a convenient way to apply these methods.

First, split your data into training and testing sets. It's crucial to ensure that treatment assignment is preserved in both sets, or at least that both groups are well-represented.

### Data Split


In [4]:
# Split data into training and testing sets
# Stratify by treatment to ensure both groups are present in train/test
X_train, X_test, y_train, y_test, t_train, t_test = train_test_split(
    X, y, t, test_size=0.3, random_state=42, stratify=t
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"Treatment distribution in train: {t_train.value_counts(normalize=True)}")
print(f"Treatment distribution in test: {t_test.value_counts(normalize=True)}")

NameError: name 'X' is not defined

### 2.1 S-Learner (Single Model Learner)

The S-Learner (Single-model Learner) is the simplest approach. It trains a single prediction model (e.g., Logistic Regression, Random Forest) on the entire dataset, including a feature that indicates treatment assignment. The CATE is then estimated by taking the difference between the predicted outcomes for a treated individual and a control individual, keeping all other features constant.

$\text{CATE}(x) = E[Y|X=x, T=1] - E[Y|X=x, T=0]$ where $E[Y|X=x, T]$ is modeled by a single function $f(x, T)$.

**Instructions:**
1.  Initialize an S-Learner using `sklift.models.SoloModel`. You can choose a base model (e.g., `RandomForestClassifier` or `LogisticRegression`).
2.  Fit the model on the training data.
3.  Predict uplift on the test set.

**Your Code Goes Here:**

In [5]:
# --- Your code for S-Learner ---

# Initialize a base model for S-Learner (e.g., RandomForestClassifier)
s_learner_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

# Initialize SoloModel (S-Learner)
# Note: SoloModel uses a single model to predict outcome based on features AND treatment
sm = SoloModel(s_learner_model)

# Fit the S-Learner model
sm.fit(X_train, y_train, t_train)

# Predict uplift on the test set
uplift_sm = sm.predict(X_test)

print("S-Learner Uplift Predictions (first 5):")
display(pd.Series(uplift_sm).head())

NameError: name 'X_train' is not defined

### 2.2 T-Learner (Two-Model Learner)

The T-Learner (Two-model Learner) trains two separate prediction models: one for the treated group ($M_T$) and one for the control group ($M_C$). Both models predict the outcome based on features $X$. The CATE is then estimated as the difference between the predictions from these two models.

$\text{CATE}(x) = M_T(x) - M_C(x)$

**Instructions:**
1.  Initialize two base models (e.g., `RandomForestClassifier`) for the treated and control groups.
2.  Initialize a T-Learner using `sklift.models.TwoModels`.
3.  Fit the T-Learner model.
4.  Predict uplift on the test set.

**Your Code Goes Here:**

In [6]:
# --- Your code for T-Learner ---

# Initialize base models for T-Learner
model_control = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
model_treatment = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

# Initialize TwoModels (T-Learner) with separate models provided in a list
tm = TwoModels(models=[model_control, model_treatment])

# Fit the T-Learner model
tm.fit(X_train, y_train, t_train)

# Predict uplift on the test set
uplift_tm = tm.predict(X_test)

print("T-Learner Uplift Predictions (first 5):")
display(pd.Series(uplift_tm).head())

TypeError: TwoModels.__init__() got an unexpected keyword argument 'estimator_t'

### 2.3 X-Learner (Cross-Learner)

The X-Learner is a more sophisticated meta-learner designed to address potential weaknesses of the T-Learner, especially when the sizes of the treatment and control groups are imbalanced. It consists of two stages:

**Stage 1:**
*   Train a model to predict the outcome in the control group: $M_C(x) = E[Y|X=x, T=0]$.
*   Train a model to predict the outcome in the treated group: $M_T(x) = E[Y|X=x, T=1]$.
*   Estimate *individualized treatment effects* for each group: $D_1 = Y_i - M_C(X_i)$ for treated units, and $D_0 = M_T(X_i) - Y_i$ for control units.

**Stage 2:**
*   Train a model to predict $D_0$ for the control group: $M_{D_0}(x) = E[D_0|X=x]$.
*   Train a model to predict $D_1$ for the treated group: $M_{D_1}(x) = E[D_1|X=x]$.
*   Combine these two models using a weighting function $g(x)$ (e.g., propensity score model) to get the final CATE estimate: $\text{CATE}(x) = g(x)M_{D_1}(x) + (1-g(x))M_{D_0}(x)$.

`sklift` implements a version of the X-Learner through `ClassTransformation` where you can provide base estimators.

**Instructions:**
1.  Initialize a base model (e.g., `RandomForestClassifier` or `LogisticRegression`) that will predict the outcome difference. `ClassTransformation` effectively turns the uplift problem into a binary classification problem.
2.  Initialize `sklift.models.ClassTransformation`.
3.  Fit the model.
4.  Predict uplift on the test set.

**Your Code Goes Here:**

In [7]:
# --- Your code for X-Learner (using ClassTransformation for demonstration) ---

# Initialize a base model for ClassTransformation (e.g., RandomForestClassifier)
x_learner_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)

# Initialize ClassTransformation
# This method transforms the labels to estimate uplift directly via a binary classifier.
# For a full X-Learner implementation with all stages, you might combine models manually or use specific libraries.
ct = ClassTransformation(x_learner_model)

# Fit the ClassTransformation model
ct.fit(X_train, y_train, t_train)

# Predict uplift on the test set
uplift_ct = ct.predict(X_test)

print("X-Learner (ClassTransformation) Uplift Predictions (first 5):")
display(pd.Series(uplift_ct).head())

NameError: name 'X_train' is not defined

### 2.4 R-Learner (Robuster Learner)

The R-Learner is an advanced meta-learner that aims to directly estimate the CATE by minimizing a regularized loss function. It models the outcome and propensity score in an initial step, and then uses these estimates to form a 'residual' outcome and 'residual' treatment, which are then used to train the final CATE model. This method is generally considered more robust to confounding.

$\\text{CATE}(x) = E[Y - E[Y|X]|X=x, T - E[T|X]]$

`sklift` provides `RLearner` which allows you to specify models for the outcome, propensity score, and the CATE itself.

**Instructions:**
1.  Initialize base models for the outcome (e.g., `RandomForestClassifier`), propensity score (e.g., `LogisticRegression`), and the CATE (e.g., `RandomForestClassifier` or `GradientBoostingClassifier`).
2.  Initialize an R-Learner using `sklift.models.RLearner`.
3.  Fit the R-Learner model.
4.  Predict uplift on the test set.

**Your Code Goes Here:**

In [8]:
# --- Your code for R-Learner ---

# Initialize base models for R-Learner
# Estimator for outcome (E[Y|X])
estimator_outcome = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
# Estimator for propensity score (E[T|X])
estimator_propensity = LogisticRegression(solver='liblinear', random_state=42)
# Estimator for CATE (the actual uplift model)
estimator_uplift = GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42)

# Initialize RLearner
rl = RLearner(
    estimator_outcome=estimator_outcome,
    estimator_propensity=estimator_propensity,
    estimator_uplift=estimator_uplift,
    random_state=42
)

# Fit the R-Learner model
rl.fit(X_train, y_train, t_train)

# Predict uplift on the test set
uplift_rl = rl.predict(X_test)

print("R-Learner Uplift Predictions (first 5):")
display(pd.Series(uplift_rl).head())

NameError: name 'RLearner' is not defined

## Task 3: Uplift Model Evaluation

Evaluating uplift models requires specialized metrics, as traditional classification metrics (like accuracy or AUC) are not suitable. We will focus on the Qini curve and the Area Under the Uplift Curve (AUUC).

### Qini Curve and AUUC

*   **Qini Curve:** Similar to the ROC or Lift curve, the Qini curve plots the cumulative uplift obtained by targeting a certain percentage of the population based on the model's uplift scores. A higher Qini curve indicates better performance.
*   **AUUC (Area Under Uplift Curve):** The area between the Qini curve and the random targeting line. A larger AUUC value indicates a better model.

**Instructions:**
1.  For each trained model (S-Learner, T-Learner, X-Learner, R-Learner), calculate and plot the Qini curve using `sklift.metrics.plot_qini_curve`.
2.  Calculate the AUUC for each model.
3.  Compare the performance of the different learners.

**Your Code Goes Here:**

In [9]:
# --- Your code for Uplift Model Evaluation ---

# Create a dictionary to store uplift predictions for easy iteration
uplift_preds = {
    'S-Learner': uplift_sm,
    'T-Learner': uplift_tm,
    'X-Learner (ClassTransformation)': uplift_ct,
    'R-Learner': uplift_rl
}

plt.figure(figsize=(10, 7))

# Plot Qini curve for each model
for name, uplift_scores in uplift_preds.items():
    plot_qini_curve(y_test, uplift_scores, t_test, perfect=False, name=name)

plt.title('Qini Curves for Different Uplift Models')
plt.legend(loc='lower right')
plt.show()

# Calculate and print AUUC for each model
print("\nAUUC Scores:")
for name, uplift_scores in uplift_preds.items():
    _, uplift_max = qini_curve(y_test, uplift_scores, t_test)
    auuc_score = uplift_at_k(y_test, uplift_scores, t_test, strategy='overall', k=1.0) # AUUC is uplift_at_k for k=1.0
    print(f"{name}: {auuc_score:.4f}")

# Additional comparison: uplift_at_k for top K%
# Example: Uplift for the top 30% of customers targeted by each model
print("\nUplift @ 30% of population:")
for name, uplift_scores in uplift_preds.items():
    uplift_30 = uplift_at_k(y_test, uplift_scores, t_test, strategy='overall', k=0.3)
    print(f"{name}: {uplift_30:.4f}")

NameError: name 'uplift_sm' is not defined

## Task 4: Interpretation and Business Recommendations

Based on your model evaluation, discuss the following:

1.  **Which model performed best?** Justify your answer using the Qini curves and AUUC scores.
2.  **Why do you think this model performed best (or worst)?** Consider the underlying assumptions and mechanisms of each learner.
3.  **Actionable Insights:** Imagine you are a marketing manager. Based on the best-performing model, how would you segment your customers? Who would you target with the treatment, and who would you avoid? Provide concrete recommendations.
4.  **Limitations:** Discuss the limitations of your current analysis and potential next steps for improving the uplift model (e.g., more complex features, different base learners, larger dataset).

**Your Discussion Goes Here (in a new Markdown cell):**

## Submission

### Project Repository Structure:
Your submission should be organized in a GitHub repository with the following structure:

```
your_github_username/uplift_modeling_project/
├── notebook.ipynb         # Your completed Colab notebook
├── README.md              # Project description, instructions to run, results summary
├── requirements.txt       # List of all Python packages used
└── data/                  # Optional: if you used additional data, place it here
```

**GitHub Requirements:**
*   The repository must be public.
*   Ensure your `notebook.ipynb` is clean, runnable, and all outputs are visible.
*   The `README.md` should clearly explain:
    *   The project's objective.
    *   How to set up the environment and run the notebook (e.g., `pip install -r requirements.txt`).
    *   A summary of your findings and business recommendations (from Task 4).

### ZIP Archive Submission:
In addition to the GitHub repository, please create a ZIP archive of your entire project folder (the `uplift_modeling_project` folder). This ZIP file should contain all the files mentioned above.

### LLM Usage Disclosure:
If you used any Large Language Models (LLMs) (e.g., ChatGPT, Gemini, Copilot) to assist with code generation, problem-solving, or conceptual understanding, you **must** disclose it clearly within your `README.md` file. For each instance of LLM usage:

*   **Specify the LLM used:** (e.g., Gemini 1.5 Pro, ChatGPT 4, GitHub Copilot).
*   **Provide the exact prompt(s) you used.**
*   **Explain how the LLM's output was used/integrated** into your solution.
*   **Describe any problems or challenges encountered** when using the LLM (e.g., incorrect code, hallucinations, difficulty in refining prompts, ethical considerations).

This is a crucial part of your submission to acknowledge the tools used in your learning process.

### Project Defense (15-Minute Online Presentation):
Finally, you will have a 15-minute online project defense. During this session, you will be expected to:

*   Briefly present your methodology and key findings.
*   Explain your choice of models and evaluation metrics.
*   Discuss your business recommendations and insights.
*   Answer questions from the lecturer regarding your code, analysis, and understanding of uplift modeling concepts.